# 07 - Appendix: Tips, Limits, and Troubleshooting

**Objective:** This reference notebook covers common pitfalls, hardware limits, debugging techniques, and advanced parameters for tProc v2 programs. Use this as a reference when things don't work as expected.

## 1. Setup

In [ ]:
# Standard imports
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import logging

from qick import *
from qick.asm_v2 import AveragerProgramV2, QickSweep1D, AsmV2

# Setup logging (set to DEBUG for detailed output)
logging.basicConfig(level=logging.INFO, format='%(levelname)-8s [%(filename)s:%(lineno)d] %(message)s')
logging.getLogger("qick_processor").setLevel(logging.WARNING)

# Connect to the board (adjust the path to your firmware)
BITSTREAM_PATH = '/path/to/your/firmware.bit'  # <--- CHANGE THIS
soc = QickSoc(BITSTREAM_PATH)
soccfg = soc

# Define hardware channels
GEN_CH = 0
RO_CH = 0

## 2. tProc Hardware Limits

Understanding the tProc's limitations will help you design programs that fit within hardware constraints.

In [ ]:
# Display tProc configuration from firmware
print("=== tProc Configuration ===")
if hasattr(soccfg, 'tproc'):
    print(f"Program Memory: {soccfg.tproc['pmem_len']} words")
    print(f"Data Memory: {soccfg.tproc['dmem_len']} words")
    print(f"Waveform Memory: {soccfg.tproc['wmem_len']} words")
    print(f"Core Clock: {soccfg.tproc['core_clk']} MHz")
    print(f"Timing Clock: {soccfg.tproc['timing_clk']} MHz")
else:
    print("tProc info not available in soccfg")

### 2.1 Program Memory Limit (4096 words)

Each tProc instruction occupies 72 bits (9 bytes). Complex programs with many loops and sweeps can approach this limit.

**How to check program size:**

In [ ]:
# Create a simple program to check its size
class SizeTestProgram(AveragerProgramV2):
    def _initialize(self, cfg):
        self.declare_gen(ch=cfg['gen_ch'], nqz=1)
        self.declare_readout(ch=cfg['ro_ch'], length=cfg['ro_len'])
        self.add_pulse(ch=cfg['gen_ch'], name="test", style="const",
                       freq=100, length=0.1, phase=0, gain=1.0)
        self.add_readoutconfig(ch=cfg['ro_ch'], name="ro", freq=100, gen_ch=cfg['gen_ch'])
        self.send_readoutconfig(ch=cfg['ro_ch'], name="ro", t=0)
    def _body(self, cfg):
        self.pulse(ch=cfg['gen_ch'], name="test", t=0)
        self.trigger(ros=[cfg['ro_ch']], pins=[0], t=0.25)

config = {'gen_ch': GEN_CH, 'ro_ch': RO_CH, 'ro_len': 0.5}
prog = SizeTestProgram(soccfg, reps=1, final_delay=0.5, cfg=config)
prog.compile()

print(f"Program size: {len(prog.binprog['pmem'])} instructions")
print(f"Memory usage: {len(prog.binprog['pmem']) / soccfg.tproc['pmem_len'] * 100:.1f}%")

### 2.2 Label Limit (2048 instruction offset)

Jump instructions use 11 bits for address literals, limiting forward jumps to 2048 instructions. This affects loops and conditionals.

**Workarounds:**
- Keep your program compact
- Use `reps_innermost=True` for large sweeps (loops are nested differently)
- Break large programs into multiple smaller ones

### 2.3 Time Limits (32-bit signed integers)

Timing parameters use 32-bit signed integers. The maximum time depends on the timing clock frequency.

```python
timing_clk_MHz = soccfg.tproc['timing_clk']  # e.g., 491.52 MHz
max_time_us = (2**31 - 1) / timing_clk_MHz
print(f"Maximum delay: {max_time_us:.0f} µs ({max_time_us/1e6:.1f} seconds)")
```

For most experiments, this is not a limiting factor.

## 3. Common Errors and Solutions

### 3.1 Pulse Collision Warnings

**Warning:** `Pulse {name} overlaps with previous pulse`

**Cause:** A pulse is scheduled before the previous pulse has finished.

**Solution:** Use `delay_auto()` to ensure proper spacing.

```python
# Wrong:
self.pulse(ch=gen_ch, name="pulse1", t=0)
self.pulse(ch=gen_ch, name="pulse2", t=0.3)  # May overlap

# Correct:
self.pulse(ch=gen_ch, name="pulse1", t=0)
self.delay_auto(gap=0.1)  # Wait for pulse1 to finish + 0.1 µs
self.pulse(ch=gen_ch, name="pulse2", t=0)
```

### 3.2 Readout Latency Mismatch

**Symptom:** Feedback readout values don't match buffered readout values.

**Cause:** `read_wait` is too short; the tProc reads stale data.

**Solution:** Increase `read_wait` until values match (see Notebook 04).

### 3.3 Sweep Value Quantization

**Symptom:** Swept values don't match the requested range exactly.

**Cause:** Hardware parameters have limited precision (e.g., gain is 15-bit, phase is 32-bit).

**Solution:** Use `get_pulse_param()` to retrieve actual values for accurate analysis.

In [ ]:
# Example: Check quantization effects
requested_gains = np.linspace(0, 1, 10)

# In a real program, you would do:
# actual_gains = prog.get_pulse_param('pulse', 'gain', as_array=True)

# Simulate quantization (15-bit gain)
max_gain_int = 32766
actual_gains = np.round(requested_gains * max_gain_int) / max_gain_int

print("Requested vs Actual Gains:")
for req, act in zip(requested_gains[:5], actual_gains[:5]):
    print(f"  {req:.4f} -> {act:.4f}")

### 3.4 Buffer Overflow

**Symptom:** Data appears truncated or corrupted.

**Cause:** More data was captured than the buffer can hold.

**Solution:** Increase buffer size or reduce acquisition length.

```python
# Estimate required buffer size
reps = 100
sweep_steps = 50
readout_samples = 256  # decimated samples
total_samples = reps * sweep_steps * readout_samples
required_transfers = np.ceil(total_samples / 256)
print(f"Required transfers: {required_transfers}")
```

## 4. Debugging Techniques

### 4.1 Enable Detailed Logging

Set logging level to DEBUG to see detailed assembler output.

In [ ]:
# Enable debug logging
logging.getLogger("qick.tprocv2_assembler").setLevel(logging.DEBUG)

# Run a simple program to see debug output
class DebugProgram(AveragerProgramV2):
    def _initialize(self, cfg):
        self.declare_gen(ch=cfg['gen_ch'], nqz=1)
        self.declare_readout(ch=cfg['ro_ch'], length=cfg['ro_len'])
        self.add_pulse(ch=cfg['gen_ch'], name="test", style="const",
                       freq=100, length=0.1, phase=0, gain=1.0)
        self.add_readoutconfig(ch=cfg['ro_ch'], name="ro", freq=100, gen_ch=cfg['gen_ch'])
        self.send_readoutconfig(ch=cfg['ro_ch'], name="ro", t=0)
    def _body(self, cfg):
        self.pulse(ch=cfg['gen_ch'], name="test", t=0)
        self.trigger(ros=[cfg['ro_ch']], pins=[0], t=0.25)

config = {'gen_ch': GEN_CH, 'ro_ch': RO_CH, 'ro_len': 0.5}
prog = DebugProgram(soccfg, reps=1, final_delay=0.5, cfg=config)

# Reset logging to INFO after debugging
logging.getLogger("qick.tprocv2_assembler").setLevel(logging.WARNING)

### 4.2 Inspect Program Structure

Use `print(prog)` to see the compiled program structure.

In [ ]:
print(prog)

### 4.3 Check Specific Macros

Use `get_macro(tag)` to inspect individual instructions.

In [ ]:
# Add tags to track specific instructions
class TaggedProgram(AveragerProgramV2):
    def _initialize(self, cfg):
        self.declare_gen(ch=cfg['gen_ch'], nqz=1)
        self.declare_readout(ch=cfg['ro_ch'], length=cfg['ro_len'])
        self.add_pulse(ch=cfg['gen_ch'], name="test", style="const",
                       freq=100, length=0.1, phase=0, gain=1.0)
        self.add_readoutconfig(ch=cfg['ro_ch'], name="ro", freq=100, gen_ch=cfg['gen_ch'])
        self.send_readoutconfig(ch=cfg['ro_ch'], name="ro", t=0)
    def _body(self, cfg):
        self.pulse(ch=cfg['gen_ch'], name="test", t=0, tag='my_pulse')
        self.delay(0.2, tag='my_delay')
        self.trigger(ros=[cfg['ro_ch']], pins=[0], t=0.25, tag='my_trigger')

prog_tagged = TaggedProgram(soccfg, reps=1, final_delay=0.5, cfg=config)

print("Pulse macro:", prog_tagged.get_macro('my_pulse'))
print("Delay macro:", prog_tagged.get_macro('my_delay'))
print("Trigger macro:", prog_tagged.get_macro('my_trigger'))

### 4.4 Verify Pulse Parameters

Use `get_pulse_param()` to check actual parameters after quantization.

In [ ]:
# In a sweep program, you can check all swept values
class SweepCheckProgram(AveragerProgramV2):
    def _initialize(self, cfg):
        self.declare_gen(ch=cfg['gen_ch'], nqz=1)
        self.declare_readout(ch=cfg['ro_ch'], length=cfg['ro_len'])
        self.add_loop("sweep", cfg['steps'])
        self.add_pulse(ch=cfg['gen_ch'], name="test", style="const",
                       freq=cfg['freq'], length=0.1,
                       phase=QickSweep1D("sweep", -180, 180),
                       gain=QickSweep1D("sweep", 0, 1))
        self.add_readoutconfig(ch=cfg['ro_ch'], name="ro", freq=cfg['freq'], gen_ch=cfg['gen_ch'])
        self.send_readoutconfig(ch=cfg['ro_ch'], name="ro", t=0)
    def _body(self, cfg):
        self.pulse(ch=cfg['gen_ch'], name="test", t=0)
        self.trigger(ros=[cfg['ro_ch']], pins=[0], t=0.25)

config_check = {'gen_ch': GEN_CH, 'ro_ch': RO_CH, 'freq': 100, 'steps': 5, 'ro_len': 0.5}
prog_check = SweepCheckProgram(soccfg, reps=1, final_delay=0.5, cfg=config_check)

phases = prog_check.get_pulse_param('test', 'phase', as_array=True)
gains = prog_check.get_pulse_param('test', 'gain', as_array=True)

print(f"Phase values: {phases}")
print(f"Gain values: {gains}")

## 5. Advanced Parameters

### 5.1 The `reps_innermost` Parameter

By default, the `reps` loop (experiment repetitions) is the **outermost** loop. This reduces sensitivity to slow drifts.

In [ ]:
class LoopOrderProgram(AveragerProgramV2):
    def _initialize(self, cfg):
        self.declare_gen(ch=cfg['gen_ch'], nqz=1)
        self.declare_readout(ch=cfg['ro_ch'], length=cfg['ro_len'])
        self.add_loop("sweep", 3)
        self.add_pulse(ch=cfg['gen_ch'], name="test", style="const",
                       freq=100, length=0.1, phase=0,
                       gain=QickSweep1D("sweep", 0, 1))
        self.add_readoutconfig(ch=cfg['ro_ch'], name="ro", freq=100, gen_ch=cfg['gen_ch'])
        self.send_readoutconfig(ch=cfg['ro_ch'], name="ro", t=0)
    def _body(self, cfg):
        self.pulse(ch=cfg['gen_ch'], name="test", t=0)
        self.trigger(ros=[cfg['ro_ch']], pins=[0], t=0.25)

config_loop = {'gen_ch': GEN_CH, 'ro_ch': RO_CH, 'ro_len': 0.5}

# Default: reps outermost
prog_default = LoopOrderProgram(soccfg, reps=4, final_delay=0.5, cfg=config_loop, reps_innermost=False)

# Alternative: reps innermost
prog_innermost = LoopOrderProgram(soccfg, reps=4, final_delay=0.5, cfg=config_loop, reps_innermost=True)

print("Default (reps outermost): Sweep completes all steps before next repetition")
print("Alternative (reps innermost): All repetitions at each sweep step before moving to next step")

### 5.2 The `final_delay` and `initial_delay` Parameters

These parameters add automatic delays at the beginning and end of each shot.

In [ ]:
# initial_delay: added after _initialize() completes
# final_delay: added at the end of each shot before the next repetition

prog_with_delays = LoopOrderProgram(soccfg, reps=2, final_delay=10.0, 
                                   cfg=config_loop, initial_delay=5.0)
print("Program with 5 µs initial delay and 10 µs final delay per shot")

## 6. Quick Reference: Parameter Quantization

| Parameter | Units | Resolution | Range |
| :--- | :--- | :--- | :--- |
| Frequency | MHz | Depends on DDS bit-width | ±DDS_range |
| Phase | degrees | 360/2^32 ≈ 8.4e-8° | 0-360° (wraps) |
| Gain | linear | 1/32766 ≈ 3e-5 | 0 to 1 |
| Time/Duration | µs | 1/timing_clock (≈2 ns) | ±2^31 ticks |
| Length (const pulse) | µs | Same as time | >0 |
| Length (flat-top flat) | µs | Same as time | >0 |

## 7. Troubleshooting Checklist

When an experiment doesn't work as expected, check these items in order:

1. **Hardware Connection** - Is the board connected? Does `soccfg` show correct configuration?
2. **Firmware Version** - Is the firmware compatible with your QICK version?
3. **Channel Configuration** - Are the generator and readout channels correct?
4. **Pulse Parameters** - Are frequency, gain, and length within valid ranges?
5. **Timeline** - Are pulses colliding? Use `delay_auto()` for spacing.
6. **Readout Latency** - Is `read_wait` properly tuned for feedback?
7. **Buffer Arm** - Are buffers armed before running? Do they have enough capacity?
8. **Program Size** - Is the program exceeding tProc memory limits?
9. **Logging** - Set logging to DEBUG and look for warnings or errors.
10. **Data Retrieval** - Are you using the correct time axis for your buffer type?

## 8. Summary

This reference notebook covers:
- Hardware limits of the tProc (program memory, labels, time ranges)
- Common errors and their solutions
- Debugging techniques (logging, program inspection, parameter verification)
- Advanced parameters (`reps_innermost`, `final_delay`, `initial_delay`)
- Parameter quantization reference table
- Troubleshooting checklist

**Next steps:**
- Review the other notebooks in this series for detailed examples
- Refer to the [QICK Documentation](https://qick-docs.readthedocs.io/) for complete API reference
- Join the QICK community for support